In [3]:
import requests
import pandas as pd
import time
import pycountry

def get_iso_code(country_name):
    """
    Safely converts full country names from the THE dataset 
    into 2-letter ISO codes required by the OpenAlex API.
    """
    # Manual overrides for common names that mismatch official ISO registries
    overrides = {
        "United Kingdom": "GB",
        "United States": "US",
        "South Korea": "KR",
        "Russia": "RU",
        "Taiwan": "TW",
        "Iran": "IR",
        "Vietnam": "VN",
        "Macao": "MO",
        "Hong Kong": "HK"
    }
    if country_name in overrides:
        return overrides[country_name]
    
    try:
        # Perform a fuzzy search in the pycountry database
        result = pycountry.countries.search_fuzzy(country_name)
        return result[0].alpha_2
    except:
        return None

def fetch_openalex_global(df_master, max_pages_per_country=20):
    """
    Loops through every unique country in the master dataset and fetches
    up to 1,000 institutions (20 pages * 50) per country from OpenAlex.
    """
    unique_countries = df_master['Country'].dropna().unique()
    print(f"🌍 Found {len(unique_countries)} unique countries in master dataset. Starting global fetch...\n")
    
    headers = {"User-Agent": "mailto:sapientia_project@example.com"}
    url = "https://api.openalex.org/institutions"
    all_institutions = []
    
    for country in unique_countries:
        iso_code = get_iso_code(country)
        
        if not iso_code:
            print(f"⚠️ Skipping '{country}': Could not find matching ISO-2 code.")
            continue
            
        print(f"Fetching data for: {country} ({iso_code})...")
        
        for page in range(1, max_pages_per_country + 1):
            params = {
                "filter": f"country_code:{iso_code},type:education",
                "per-page": 50,
                "page": page
            }
            
            response = requests.get(url, params=params, headers=headers)
            
            if response.status_code != 200:
                print(f"   API Error {response.status_code} for {country}. Moving to next country.")
                break
                
            data = response.json()
            results = data.get("results", [])
            
            if not results:
                break # Reached the end of the institutions for this country
                
            for inst in results:
                all_institutions.append({
                    "openalex_id": inst.get("id"),
                    "university_name": inst.get("display_name"),
                    "country_code": iso_code,
                    "original_country_name": country, # Keep this to make Step 2 (Fuzzy Matching) easier!
                    "total_works": inst.get("works_count", 0),
                    "total_citations": inst.get("cited_by_count", 0)
                })
            
            time.sleep(0.1) # Be polite to the API rate limits
            
    df_openalex = pd.DataFrame(all_institutions)
    print(f"\n✅ Global OpenAlex data retrieval complete! Total institutions fetched: {len(df_openalex)}")
    return df_openalex

# --- EXECUTION SCRIPT ---

# 1. Load the master dataset to get the country list
df_master = pd.read_csv('final_merged_dataset_with_RD.csv')

# 2. Execute the global fetch (this will take a few minutes to run through the whole world!)
df_bibliometrics_global = fetch_openalex_global(df_master, max_pages_per_country=20)

# 3. Save it to a CSV
df_bibliometrics_global.to_csv("openalex_bibliometrics_global.csv", index=False)

print(df_bibliometrics_global.head())

🌍 Found 115 unique countries in master dataset. Starting global fetch...

Fetching data for: United Kingdom (GB)...
Fetching data for: United States (US)...
Fetching data for: Switzerland (CH)...
Fetching data for: China (CN)...
Fetching data for: Singapore (SG)...
Fetching data for: Canada (CA)...
Fetching data for: Japan (JP)...
Fetching data for: Germany (DE)...
Fetching data for: Hong Kong (HK)...
Fetching data for: Australia (AU)...
Fetching data for: Belgium (BE)...
Fetching data for: France (FR)...
Fetching data for: Sweden (SE)...
Fetching data for: Netherlands (NL)...
Fetching data for: South Korea (KR)...
Fetching data for: Denmark (DK)...
Fetching data for: Austria (AT)...
Fetching data for: Finland (FI)...
Fetching data for: Norway (NO)...
Fetching data for: Italy (IT)...
Fetching data for: Russian Federation (RU)...
Fetching data for: Taiwan (TW)...
Fetching data for: Macao (MO)...
Fetching data for: Spain (ES)...
Fetching data for: New Zealand (NZ)...
Fetching data for: S

In [4]:
import pandas as pd
from thefuzz import fuzz
from thefuzz import process

def global_partitioned_fuzzy_match(df_master, df_external, master_col='Name', ext_col='university_name', threshold=85):
    """
    Performs probabilistic record linkage by partitioning the data by country.
    This drops computation time by 99% and prevents cross-country false positives.
    """
    print(f"Starting global partitioned fuzzy matching...")
    print(f"Master records: {len(df_master)} | External records: {len(df_external)}")
    
    matched_records = []
    
    # Get a list of unique countries in the master dataset
    unique_countries = df_master['Country'].dropna().unique()
    
    for country in unique_countries:
        # 1. Isolate the master records for this specific country
        master_subset = df_master[df_master['Country'] == country]
        
        # 2. Isolate the OpenAlex records for this specific country
        # We rely on the 'original_country_name' column we generated in Step 1
        if 'original_country_name' in df_external.columns:
            ext_subset = df_external[df_external['original_country_name'] == country]
        else:
            print(f"⚠️ Warning: 'original_country_name' missing in external data. Check Step 1!")
            ext_subset = pd.DataFrame()
            
        external_names = ext_subset[ext_col].dropna().tolist()
        
        # Only print if we are actually processing a batch
        if not master_subset.empty:
             print(f"Matching {len(master_subset)} institutions in: {country} (Against {len(external_names)} OpenAlex records)...")
        
        # 3. Perform matching ONLY within this specific country subset
        for idx, row in master_subset.iterrows():
            master_name = row[master_col]
            combined_row = row.to_dict()
            
            # Default state (no match found)
            combined_row['openalex_match_name'] = None
            combined_row['fuzzy_match_score'] = None
            combined_row['openalex_works'] = None
            combined_row['openalex_citations'] = None
            
            # If we successfully pulled OpenAlex data for this country, attempt a match
            if external_names:
                best_match, score = process.extractOne(master_name, external_names, scorer=fuzz.token_sort_ratio)
                
                if score >= threshold:
                    ext_data = ext_subset[ext_subset[ext_col] == best_match].iloc[0]
                    combined_row['openalex_match_name'] = best_match
                    combined_row['fuzzy_match_score'] = score
                    combined_row['openalex_works'] = ext_data['total_works']
                    combined_row['openalex_citations'] = ext_data['total_citations']
            
            matched_records.append(combined_row)

    # Convert the results back into a DataFrame
    df_merged = pd.DataFrame(matched_records)
    
    match_count = df_merged['openalex_match_name'].notna().sum()
    print(f"\n✅ Global Fuzzy Matching Complete. Successfully linked {match_count} records out of {len(df_master)}.")
    
    return df_merged

# --- EXECUTION SCRIPT ---

# 1. Load your master THE dataset
df_master = pd.read_csv('final_merged_dataset_with_RD.csv')

# 2. Load the GLOBAL OpenAlex data we pulled in Step 1
try:
    df_openalex = pd.read_csv('openalex_bibliometrics_global.csv')
    
    # 3. Run the Partitioned Probabilistic Record Linkage
    df_final_linked = global_partitioned_fuzzy_match(df_master, df_openalex, master_col='Name', ext_col='university_name', threshold=85)
    
    # Save the output
    df_final_linked.to_csv('master_with_openalex_linked.csv', index=False)
    
    # Show the professor a sample of successful matches
    print("\nSample of successful global fuzzy matches:")
    successful_matches = df_final_linked[df_final_linked['fuzzy_match_score'] >= 85]
    print(successful_matches[['Name', 'Country', 'openalex_match_name', 'fuzzy_match_score']].head())

except FileNotFoundError:
    print("Error: Could not find 'openalex_bibliometrics_global.csv'. Please run the Global Step 1 first!")

Starting global partitioned fuzzy matching...
Master records: 2191 | External records: 16825
Matching 109 institutions in: United Kingdom (Against 578 OpenAlex records)...
Matching 171 institutions in: United States (Against 1000 OpenAlex records)...
Matching 12 institutions in: Switzerland (Against 152 OpenAlex records)...
Matching 97 institutions in: China (Against 1000 OpenAlex records)...
Matching 2 institutions in: Singapore (Against 40 OpenAlex records)...
Matching 34 institutions in: Canada (Against 345 OpenAlex records)...
Matching 115 institutions in: Japan (Against 1000 OpenAlex records)...
Matching 55 institutions in: Germany (Against 573 OpenAlex records)...
Matching 8 institutions in: Hong Kong (Against 14 OpenAlex records)...
Matching 37 institutions in: Australia (Against 132 OpenAlex records)...
Matching 10 institutions in: Belgium (Against 86 OpenAlex records)...
Matching 48 institutions in: France (Against 496 OpenAlex records)...
Matching 13 institutions in: Sweden (

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

def perform_louvain_clustering(df, context_cols=['GDP_per_Capita', 'R&D Expenditure (%)'], target_clusters=6):
    print(f"Starting Louvain Clustering on {len(df)} institutions...")
    
    df_clean = df.dropna(subset=context_cols).copy()
    
    # 1. Scale Context Variables
    Z_data = df_clean[context_cols].values
    scaler = StandardScaler()
    Z_scaled = scaler.fit_transform(Z_data)
    
    # Temporarily store scaled data for distance calculations later
    df_clean['Z_scaled_0'] = Z_scaled[:, 0]
    df_clean['Z_scaled_1'] = Z_scaled[:, 1]
    
    # 2. Build Similarity Network
    print("Building similarity matrix...")
    dist_matrix = pairwise_distances(Z_scaled, metric='euclidean')
    similarity_matrix = 1 / (1 + dist_matrix)
    np.fill_diagonal(similarity_matrix, 0) 
    
    # SPARSIFICATION (Keep top 10% of edges)
    nonzero_sims = similarity_matrix[similarity_matrix > 0]
    if len(nonzero_sims) > 0:
        threshold = np.percentile(nonzero_sims, 90)
        similarity_matrix[similarity_matrix < threshold] = 0
        
    G = nx.from_numpy_array(similarity_matrix)
    
    # 3. Base Louvain Algorithm
    print("Executing base Louvain algorithm...")
    partition = community_louvain.best_partition(G, weight='weight', resolution=1.0)
    
    # Map initial clusters to dataframe
    cluster_labels = [partition[i] for i in range(len(df_clean))]
    df_clean['Initial_Cluster'] = cluster_labels
    
    # 4. Centroid-Based Community Agglomeration (Forcing 6 Clusters)
    print(f"Agglomerating micro-communities to exactly {target_clusters} Core Archetypes...")
    
    # Identify the Top 6 largest clusters
    top_clusters = df_clean['Initial_Cluster'].value_counts().nlargest(target_clusters).index.tolist()
    
    # Calculate the mathematical centroid (average Z-scores) of these 6 core clusters
    centroids = {}
    for c in top_clusters:
        c_data = df_clean[df_clean['Initial_Cluster'] == c]
        centroids[c] = np.array([c_data['Z_scaled_0'].mean(), c_data['Z_scaled_1'].mean()])
        
    # Reassign minority clusters to the closest Core Centroid
    def assign_to_core(row):
        if row['Initial_Cluster'] in top_clusters:
            return row['Initial_Cluster'] # Already in a core cluster
        else:
            # Find the closest core cluster
            row_z = np.array([row['Z_scaled_0'], row['Z_scaled_1']])
            min_dist = float('inf')
            best_cluster = -1
            for c_id, centroid in centroids.items():
                dist = np.linalg.norm(row_z - centroid)
                if dist < min_dist:
                    min_dist = dist
                    best_cluster = c_id
            return best_cluster

    df_clean['Final_Cluster'] = df_clean.apply(assign_to_core, axis=1)
    
    # Clean up cluster names (Rename them 1 through 6 neatly)
    unique_final_clusters = df_clean['Final_Cluster'].unique()
    mapping = {old_id: new_id + 1 for new_id, old_id in enumerate(unique_final_clusters)}
    df_clean['Louvain_Cluster'] = df_clean['Final_Cluster'].map(mapping)
    
    # 5. Recalculate Final Modularity (Q)
    new_partition = {i: int(df_clean['Louvain_Cluster'].iloc[i]) for i in range(len(df_clean))}
    final_modularity = community_louvain.modularity(new_partition, G, weight='weight')
    print(f"✅ Agglomeration complete! Final Modularity (Q) = {final_modularity:.4f}")
    
    # Clean up temporary columns
    df_clean = df_clean.drop(columns=['Z_scaled_0', 'Z_scaled_1', 'Initial_Cluster', 'Final_Cluster'])
    
    # Merge back to original dataframe
    df_final = df.merge(df_clean[['Name', 'Louvain_Cluster']], on='Name', how='left')
    
    return df_final, final_modularity

# --- EXECUTION SCRIPT ---
try:
    df_linked = pd.read_csv('master_with_openalex_linked.csv')
    
    df_clustered, final_q = perform_louvain_clustering(
        df_linked, 
        context_cols=['GDP_per_Capita', 'R&D Expenditure (%)'],
        target_clusters=6
    )
    
    df_clustered.to_csv('master_with_clusters.csv', index=False)
    
    print("\nFinal Cluster Summary (Number of institutions per cluster):")
    print(df_clustered['Louvain_Cluster'].value_counts().sort_index())

except FileNotFoundError:
    print("Error: Could not find 'master_with_openalex_linked.csv'. Please run Step 2 first!")

Starting Louvain Clustering on 2191 institutions...
Building similarity matrix...
Executing base Louvain algorithm...
Agglomerating micro-communities to exactly 6 Core Archetypes...
✅ Agglomeration complete! Final Modularity (Q) = 0.6571

Final Cluster Summary (Number of institutions per cluster):
Louvain_Cluster
1.0    258
2.0    208
3.0    220
4.0    363
5.0    268
6.0    513
Name: count, dtype: int64


In [2]:
import pandas as pd
import numpy as np
import rpy2.robjects as robjects
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

def conditional_order_m(df, input_col='R&D Expenditure (%)', output_col='Research Quality', cond_col='Louvain_Cluster', m_size=50):
    """
    Calculates Order-m efficiency conditional on the discrete macroeconomic 
    peer clusters (Louvain) using the R 'nonparaeff' package.
    """
    print(f"Starting Conditional Order-m calculation (m={m_size}) across {df[cond_col].nunique()} peer clusters...")
    
    # 1. Clean data: Remove rows with missing inputs, outputs, or clusters
    df_clean = df.dropna(subset=[input_col, output_col, cond_col]).copy()
    
    # 2. Prevent zeros (Order-m requires strictly positive inputs/outputs)
    df_clean[input_col] = df_clean[input_col].apply(lambda x: max(x, 0.001))
    df_clean[output_col] = df_clean[output_col].apply(lambda x: max(x, 0.001))
    
    # 3. Import the correct R package (nonparaeff handles order-m, not Benchmarking)
    try:
        nonparaeff = importr('nonparaeff')
    except Exception as e:
         print("❌ Error: The R package 'nonparaeff' is not installed or accessible.")
         print("Please run `install.packages('nonparaeff')` in your R console.")
         return df # Returns original dataframe without new columns!

    # Prepare an empty list to collect the scored dataframes
    scored_clusters = []
    
    # 4. Iterate through each macroeconomic cluster (The "Conditional" aspect)
    for cluster_id, group in df_clean.groupby(cond_col):
        print(f"   Processing Cluster {int(cluster_id)} (n={len(group)})...")
        
        # Safety check: 'm' cannot be larger than the number of institutions in the cluster
        actual_m = min(m_size, len(group) - 1)
        if actual_m < 2:
            print(f"      ⚠️ Cluster {int(cluster_id)} is too small for Order-m. Skipping.")
            continue
            
        # Convert pandas columns to DataFrames inside R list structure
        with localconverter(robjects.default_converter + pandas2ri.converter):
            # nonparaeff.orderm() requires data.frames with inputs and outputs combined
            xy_df = pd.DataFrame({
                'input': group[input_col].values,
                'output': group[output_col].values
            })
            
            xy_r = pandas2ri.py2rpy(xy_df)
            
            try:
                # R function: orderm(base, frontier, noutput, orientation, M, B)
                # orientation = 2 (output-oriented), noutput = 1
                order_m_result = nonparaeff.orderm(base=xy_r, frontier=xy_r, noutput=1, orientation=2, M=actual_m)
                
                # Extract scores (localconverter auto-converts the resulting R data.frame back to a pandas DataFrame)
                efficiency_scores = order_m_result['eff'].values
                
                # Invert and normalize to 0-100 scale
                group = group.copy()
                group['Order_M_Efficiency_Raw'] = efficiency_scores
                group['Sapientia_Efficiency_Score'] = (1.0 / group['Order_M_Efficiency_Raw']) * 100
                
                # Cap at 100 to handle super-efficient mathematical outliers
                group['Sapientia_Efficiency_Score'] = group['Sapientia_Efficiency_Score'].clip(upper=100)
                group['Sapientia_Efficiency_Score'] = group['Sapientia_Efficiency_Score'].round(2)
                
                scored_clusters.append(group)
                
            except Exception as e:
                print(f"      ❌ Error during R execution for Cluster {int(cluster_id)}: {e}")

    if not scored_clusters:
        return df
        
    # 5. Recombine the conditional frontiers into one dataset
    df_scored = pd.concat(scored_clusters)
    print("✅ Conditional Order-m calculation complete!")
    
    # Merge results back into the main DataFrame
    df_final = df.merge(
        df_scored[['Name', 'Order_M_Efficiency_Raw', 'Sapientia_Efficiency_Score']], 
        on='Name', 
        how='left'
    )
    return df_final

# --- EXECUTION SCRIPT ---

try:
    # Load the clustered dataset from Step 3
    df_clustered = pd.read_csv('master_with_clusters.csv')
    
    # Run the conditional Order-m pipeline
    df_final_sapientia = conditional_order_m(
        df_clustered, 
        input_col='R&D Expenditure (%)', 
        output_col='Research Quality',
        cond_col='Louvain_Cluster',
        m_size=50
    )
    
    # Save the absolute final dataset
    df_final_sapientia.to_csv('final_sapientia_master.csv', index=False)
    
    # Check if the score column was actually generated before sorting
    if 'Sapientia_Efficiency_Score' in df_final_sapientia.columns:
        print("\n🏆 Top 5 Most Efficient Universities (Global Frontier):")
        top_efficient = df_final_sapientia.sort_values(by='Sapientia_Efficiency_Score', ascending=False).head(5)
        print(top_efficient[['Name', 'Country', 'Louvain_Cluster', 'Sapientia_Efficiency_Score']])
    else:
        print("\n⚠️ Note: The 'Sapientia_Efficiency_Score' column was not generated.")
        print("This usually happens if the R package is missing or failing.")

except FileNotFoundError:
    print("Error: Could not find 'master_with_clusters.csv'. Please run Step 3 first!")

Starting Conditional Order-m calculation (m=50) across 6 peer clusters...
   Processing Cluster 1 (n=258)...
   Processing Cluster 2 (n=208)...
   Processing Cluster 3 (n=220)...
   Processing Cluster 4 (n=363)...
   Processing Cluster 5 (n=268)...
   Processing Cluster 6 (n=513)...
✅ Conditional Order-m calculation complete!

🏆 Top 5 Most Efficient Universities (Global Frontier):
                                  Name        Country  Louvain_Cluster  \
273       University of Nebraska Omaha  United States              2.0   
707                    Umeå University         Sweden              1.0   
700  KTH Royal Institute of Technology         Sweden              1.0   
701                 Uppsala University         Sweden              1.0   
702               Stockholm University         Sweden              1.0   

     Sapientia_Efficiency_Score  
273                       100.0  
707                       100.0  
700                       100.0  
701                       100.0  
7